# Detect extrusions (or another event) on a movie

Used already trained neural network(s) to detect extrusions or other events from movie(s).

DeXtrusion calculates a probability map which reflects the probability of having an extrusion (or other event) for each pixel at each time of the movie. The probability map can be converted to single point position as ImageJ ROI. The position of an event will be at the centroid of a high probability volume.

## Initialization

In [1]:
import os
from dextrusion.DeXtrusion import DeXtrusion
from dextrusion.DialogDeXtrusion import DialogDeXtrusion

# default parameters
talkative = True  ## print info messages
dexter = DeXtrusion(verbose=talkative)

imname = "../data/notum_mix"    ## folder where data are saved to initialize the file browser
imname = "~/Proj/IAH/HackatonAppose/dextrusion-appose/data"
modeldir = os.path.join(os.path.abspath(os.path.join(os.getcwd(), os.pardir)), "DeXNets", "notum_all")  ## folder where models are saved to initialize the file browser
groupsize = 150000       ## number of windows that are runned through the neural network at the same time - depends on the computer processing capabilities
                         ## higher will run faster, but too high can fail

2026-03-25 16:25:39.140414: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-25 16:25:39.230077: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-25 16:25:39.795103: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvrtc.so.11.1: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /home/gaelle/Envs/miniforge3/envs/dextrusenv/lib/python3.10/site-packages/cv2/../../li

Tensorflow with Cuda: True
Tensorflow version: 2.11.0
Keras version: Unknown
Using Keras 3 API: False
Num GPUs Available:  1


### Graphical interface pop-up to choose the detection parameters

In [2]:
# Choose parameters through graphical interface
diag = DialogDeXtrusion()
diag.dialog_main(modeldir, imname)

## typical cell diameter in the movie. Used to resize the movie if necessary to match with the reference scale used in the training.
cell_diameter = float(diag.cell_diameter)          
## typical duration of an extrusion event (number of frames it is visible). Used to resize the movie to the reference scale if necessary
extrusion_duration = float(diag.extrusion_duration) 
## xy step used to move the sliding windows
shift_xy = int(diag.dxyval)
## frame interval used to move the sliding windows
shift_t = int(diag.dtval)
## save the resulting probability maps (of all events)
save_proba = bool(diag.saveProba)
## save the cleaned (thresholded) probability map of the selected event
save_proba_one = bool(diag.saveProbaOne)
## save the ROIs file of all the events
save_rois = bool(diag.saveRois)
## Choose which event to save the probamap of. (Put None to do all events)
cat = int(diag.event_ind)
## Names of the possible events
catnames = diag.event
## min average probability in volume to keep ROI
roi_threshold = float(diag.threshold) 
## min positive volume to keep ROI
roi_volume = float(diag.proba_vol) 
## computational parameters, number of windows to evaluate at the time
groupsize = int(diag.group_size)       

modeldir = diag.modeldir
from glob import glob
# if folder "modeldir" is not one model directory but contains several models subdirectory, use them all
if not os.path.exists( os.path.join(modeldir,"config.cfg") ):
    model = glob(modeldir+"/*/", recursive = False)
else:
    model = [modeldir]
imname = diag.imagepath
if talkative:
    print("Using network(s) "+str(model))
    print("On movie(s) "+str(imname))

['/home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll1/', '/home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll0/']
/home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll1/
Using network(s) ['/home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll1/', '/home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll0/']
On movie(s) ['/home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/002-crop-1.tif', '/home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/002-crop.tif']


### Run the dextrusion detection on the selected movie(s) with the selected trained network(s)

In [3]:
## Merge "peaks" if distance < disxy (spatial) & dist (time) 
disxy = 10
distime = 4
# Parameters: astime: save tif image as temporal stack (else slice stack)
temporal = True

for image in imname:
    print(image)
    if os.path.exists(image):
        if talkative:
            print( "Detecting events on movie "+str(image) )
        
        ## Detect events
        dexter.detect_events_onmovie( image, models=model, 
                                     cell_diameter=cell_diameter, extrusion_duration=extrusion_duration, 
                                     dxy=shift_xy, dz=shift_t, 
                                     group_size=groupsize )
          
        ## Save probability maps to file, to visualize the results
        if save_proba:
            dexter.write_probamaps(cat=None, astime=temporal)
        
        ## Save cleaned probability map(s) to file, keeping only events above the size and probability thresholds (for visualisation) 
        if save_proba_one:
            dexter.write_cleanedprobamap( cat=cat, volume_threshold=roi_volume, proba_threshold=roi_threshold, 
                                         disxy=disxy, distime=distime, astime=temporal )
        
        ## Look for centroid of "positive" volumes in the probability maps and convert it to a point event 
        ## Keep only ROIs that have a big enough volume ('volume_threshold') and with high enough probability ('proba_threshold') 
        # - put 0 and 0 to keep all detections
        if save_rois:
            dexter.get_rois(cat=cat, volume_threshold=roi_volume, proba_threshold=roi_threshold, disxy=disxy, dist=distime)

/home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/002-crop-1.tif
Detecting events on movie /home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/002-crop-1.tif


2026-03-25 16:25:55.216178: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-25 16:25:55.753717: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1613] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 21088 MB memory:  -> device: 0, name: NVIDIA RTX A5000, pci bus id: 0000:65:00.0, compute capability: 8.6


  Initial image shape: (20, 97, 188)
  Rescaled image shape: (20, 97, 188)
  Extended image shape: (30, 97, 188)
   Doing windows with shift: 0 0 and model /home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll1/
   Number of windows to categorize: 900
     - Do one group: 0


2026-03-25 16:26:01.931710: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:428] Loaded cuDNN version 8907


23/29 [======================>.......] - ETA: 0s

2026-03-25 16:26:02.915229: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:630] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


29/29 [==============================] - 2s 10ms/step
   Doing windows with shift: 1 5 and model /home/gaelle/Proj/RL/detecExtrusion/togit/DeXtrusion/DeXNets/notum_all/notumAll0/
   Number of windows to categorize: 720
     - Do one group: 0
23/23 [==============================] - 1s 11ms/step
---- Detection finished, took 0.2173139174779256 minutes ---
Probability map /home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/results/002-crop-1_cell_death_proba.tif saved
Probability map /home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/results/002-crop-1_cell_sop_proba.tif saved
Probability map /home/gaelle/Proj/IAH/HackatonAppose/dextrusion-appose/data/results/002-crop-1_cell_division_proba.tif saved
[ImagejRoi(
    roitype=ROI_TYPE.POINT,
    name='0004-0073-0094',
    version=227,
    top=73,
    left=94,
    n_coordinates=1,
    stroke_width=3,
    stroke_color=b'\xff\xff\x00\x00',
    position=4,
    c_position=1,
    z_position=1,
    t_position=4,
    integer_coordi